# **RAG Pipeline : Data Ingestion to Vector DB Pipeline**

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
def process_pdf_files(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")
    for pdf_file in pdf_files:
        print(f"\nprocessing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source-file'] = pdf_file.name
                doc.metadata['type'] = 'pdf'
            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

pdf_directory = "../data/pdf_files"
documents = process_pdf_files(pdf_directory)

Found 1 PDF files in ../data/pdf_files

processing file: ..\data\pdf_files\osqb.pdf
Loaded 6 pages from osqb.pdf

Total documents loaded: 6


In [14]:
### Text splitting get into chunks
def text_splitter(documents,chunk_size = 1000, chunk_overlap=200):
    text_splt = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splt.split_documents(documents)
    print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")    
    if chunks:
        print(f"\nExample chunk:")
        print(f"Content: {chunks[0].page_content[:200]}...")
        print(f"Metadata: {chunks[0].metadata}")
    
    return chunks   

chunks = text_splitter(documents)

Split 6 document(s) into 15 chunks.

Example chunk:
Content: Operating Systems — Unit-wise & Topic-wise Repeated Question Bank Operating Systems — Unit-wise & Topic-wise Repeated Question Bank
UNIT 1 — INTRODUCTION & PROCESS MANAGEMENT
UNIT 1 — INTRODUCTION & P...
Metadata: {'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.4', 'creationdate': '2026-05-16T17:04:14+00:00', 'title': 'Markdown To PDF', 'source': '..\\data\\pdf_files\\osqb.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source-file': 'osqb.pdf', 'type': 'pdf'}
